# Unified Training Pipeline with Dynamic Training

This notebook integrates data preprocessing and training of a ResNet for both binary and multiclass motor fault classification. Each epoch a new training dataset is generated

In [ ]:
# Logging
import comet_ml
import yaml
import datetime

with open("../cfg.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config['API_KEY']
project_name = config['PROJECT_NAME']
workspace = config['WORKSPACE']

experiment = comet_ml.start(api_key=api_key, 
                        project_name=project_name, 
                        workspace=workspace)

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(current_time)
experiment.set_name(f"Run: {current_time}")
experiment.log_code()

In [ ]:
import numpy as np
import torch
import random

from src.data_pipeline import (
    generate_synthetic_peak, process_file, insert_synthetic_peaks, create_dataloaders, create_dataset
)
from src.electrical_signature_frequencies import ANOMALY_FREQS
from src.models import ResNet, ResidualBlock
from src.train import train_resnet_model
from src.inference import inference_resnet_model
from src.evaluation import calculate_metrics

from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device 
device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
experiment.log_parameter("device", device)
device = torch.device(device)
print("Using device:", device)

## 1. Load Engine Configuration and Set Parameters

In [ ]:
# Load training parameters from train.yml
with open("../training_configs/train.yml", "r") as f:
    train_params = yaml.safe_load(f)

# Processing Parameters
f_sampling = train_params["processing_parameters"]["f_sampling"]
shift = train_params["processing_parameters"]["shift"]
segment_length = train_params["processing_parameters"]["segment_length"]
cutoff_freq = train_params["processing_parameters"]["cutoff_freq"]
peak_type = train_params["processing_parameters"]["peak_type"]
peak_segment = train_params["processing_parameters"]["peak_segment"]
amplitude_range = tuple(train_params["processing_parameters"]["amplitude_range"])

# Training Parameters
num_epochs_binary = train_params["training_parameters"]["num_epochs_binary"]
num_epochs_multi = train_params["training_parameters"]["num_epochs_multi"]

# Data Parameters
batch_size = train_params["data_parameters"]["batch_size"]
file_label_map = train_params["data_parameters"]["file_label_map"]
fault_types_available = train_params["data_parameters"]["fault_types_available"]

# Dataset Parameters
normalization_method = train_params["dataset_parameters"]["normalization_method"]


# Load Engine Configuration
with open('../engine_configs/LIMAN.yml', 'r') as f:
    engine_config = yaml.safe_load(f)

experiment.log_parameters({
    "f_sampling": f_sampling,
    "shift": shift,
    "segment_length": segment_length,
    "peak_segment": peak_segment,
    "cutoff_freq": cutoff_freq,
    "file_label_map": file_label_map,
    "fault_types_available": fault_types_available,
    "peak_type": peak_type,
    "num_epochs_binary": num_epochs_binary,
    "num_epochs_multi": num_epochs_multi,
    "batch_size": batch_size,
    "normalization_method": normalization_method
})

fault_freqs = {
    fault_type: ANOMALY_FREQS.get(fault_type, lambda ec: [])(engine_config)
    for fault_type in fault_types_available
}
experiment.log_parameters(fault_freqs)

In [ ]:
# Saving spectrums before and after peak injection

sample_file = None
for fp, lbl in file_label_map.items():
    if lbl == 'normal':
        sample_file = fp
        break

if sample_file is not None:
    try:
        segs, labels, freqs = process_file(sample_file, engine_config, 'normal',
                                            f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                            segment_length=segment_length, peak_segment=peak_segment, step=shift,
                                            apply_window=False, db=True,
                                            synthetic_fault_fraction=0.0, fault_types=None, amplitude_range=amplitude_range)
    except NameError:
        pass

    if segs is not None and len(segs) > 0:
        sample_segment = segs[0]
        for fault_type in fault_types_available:
            injected_segment = insert_synthetic_peaks(
                segment=sample_segment,
                fft_freqs=freqs,
                fault_type=fault_type,
                engine_config=engine_config,
                peak_segment=peak_segment,
                peak_generator=generate_synthetic_peak,
                sigma=2.0,
                method='gaussian'
            )
            difference = injected_segment - sample_segment
            fig, axs = plt.subplots(1, 3, figsize=(18, 5))
            axs[0].plot(freqs, sample_segment, color='blue')
            axs[0].set_title("Before Processing")
            axs[0].set_xlabel("Frequency")
            axs[0].set_ylabel("Magnitude (dB)")
            axs[0].grid(True)
            
            axs[1].plot(freqs, injected_segment, color='orange')
            axs[1].set_title(f"After Processing ({fault_type})")
            axs[1].set_xlabel("Frequency")
            axs[1].grid(True)
            
            axs[2].plot(freqs, difference, color='green')
            axs[2].set_title("Difference (After - Before)")
            axs[2].set_xlabel("Frequency")
            axs[2].grid(True)
            
            fig.suptitle(f"Spectrum Comparison for {fault_type}")
            experiment.log_figure(figure=fig, figure_name=f"spectrum_comparison_{fault_type}.png")
            plt.close(fig)

## 2. Train the ResNet Binary Classifier

In [ ]:
# unified_datasets.py
import torch
from torch.utils.data import TensorDataset
from src.data_pipeline import create_dataset
from src.data_pipeline import (
    insert_synthetic_peaks,
    add_noise_to_segment,
    normalize_segment,
    generate_synthetic_peak
)
import numpy as np
import random

# Unified Dataset class that supports dynamic injection on the fly (for training only)
class DynamicSyntheticDataset(torch.utils.data.Dataset):
    def __init__(self, raw_segments, raw_labels, freqs, engine_config, mode="binary",
                 fault_types_available=None, peak_segment=6, amplitude_range=(1.0, 5.0),
                 normalization_method='min-max', add_noise=True, synthetic_fault_fraction=None):
        """
        Parameters:
          raw_segments (np.ndarray): Preprocessed training segments (without noise/normalization).
          raw_labels (np.ndarray): Corresponding labels.
          freqs (np.ndarray): FFT frequency bins.
          engine_config (dict): Engine configuration parameters.
          mode (str): "binary" or "multiclass".
          fault_types_available (list): List of fault types available.
          peak_segment (int): Half-window size around the fault frequency.
          amplitude_range (tuple): Range for random peak amplitudes.
          normalization_method (str): Normalization method (e.g., 'z-score' or 'min-max').
          add_noise (bool): Whether to add noise.
          synthetic_fault_fraction (float): Fraction of normal segments to inject anomalies.
                    If None, defaults to 0.5 for binary mode or k/(k+1) for multiclass.
        """
        self.raw_segments = raw_segments
        self.raw_labels = raw_labels
        self.freqs = freqs
        self.engine_config = engine_config
        self.mode = mode
        self.fault_types_available = fault_types_available
        self.peak_segment = peak_segment
        self.amplitude_range = amplitude_range
        self.normalization_method = normalization_method
        self.add_noise = add_noise

        if synthetic_fault_fraction is None:
            if mode == "binary":
                self.synthetic_fault_fraction = 0.5
            elif mode == "multiclass":
                if not fault_types_available or len(fault_types_available) == 0:
                    raise ValueError("For multiclass mode, fault_types_available must be provided and non-empty.")
                k = len(fault_types_available)
                self.synthetic_fault_fraction = k / (k + 1)
        else:
            self.synthetic_fault_fraction = synthetic_fault_fraction

    def __len__(self):
        return len(self.raw_segments)

    def __getitem__(self, idx):
        seg = self.raw_segments[idx]
        label = self.raw_labels[idx]

        # In our convention: for binary mode, "normal" is 0; for multiclass, we assume "normal" is labeled as "normal"
        is_normal = (label == 0) if self.mode == "binary" else (label == "normal")

        # For training only, inject synthetic anomalies on the fly
        if is_normal and random.random() < self.synthetic_fault_fraction:
            chosen_fault = random.choice(self.fault_types_available)
            seg = insert_synthetic_peaks(
                segment=seg,
                fft_freqs=self.freqs,
                fault_type=chosen_fault,
                engine_config=self.engine_config,
                peak_segment=self.peak_segment,
                peak_generator=generate_synthetic_peak,
                amplitude_range=self.amplitude_range,
                sigma=2.0,
                method='gaussian'
            )
            label = 1 if self.mode == "binary" else chosen_fault

        # Optionally add noise
        if self.add_noise:
            seg = add_noise_to_segment(seg)
        # Normalize the segment
        if self.normalization_method:
            seg, _ = normalize_segment(seg, method=self.normalization_method)

        # Ensure the segment is float32 and add a channel dimension
        seg = seg.astype(np.float32)
        seg = np.expand_dims(seg, axis=0)  # Shape becomes [1, L]

        return seg, label


def create_unified_datasets(file_label_map, engine_config, mode="binary", dynamic_training=True,
                            f_sampling=10000, cutoff_freq=250, segment_length=10000, peak_segment=6, step=20,
                            apply_window=False, db=True, fault_types_available=None, amplitude_range=(1.0, 5.0),
                            normalization_method='min-max', add_noise=True, test_size=0.3, val_size=0.2, seed=42):
    """
    Creates unified PyTorch datasets for training, validation, and testing.
    
    For training:
      - If dynamic_training is True, wraps the raw training segments into a UnifiedDataset
        that injects synthetic anomalies on the fly.
      - Otherwise, returns a static TensorDataset.
    For validation and test, returns static TensorDatasets.
    
    This function uses your existing create_dataset to process the raw files.
    
    Returns:
      dict: A dictionary with keys "train", "val", and "test" containing PyTorch Dataset objects.
    """
    # Generate static processed arrays using your existing create_dataset
    dataset_static = create_dataset(
        file_label_map, engine_config, mode=mode,
        f_sampling=f_sampling, cutoff_freq=cutoff_freq,
        segment_length=segment_length, peak_segment=peak_segment, step=step,
        apply_window=apply_window, db=db,
        fault_types_available=fault_types_available,
        include_real_anomalies_in_training=False,
        amplitude_range=amplitude_range,
        test_size=test_size, val_size=val_size,
        normalization_method=normalization_method,
        add_noise=add_noise,
        seed=seed
    )

    # For multiclass mode, obtain the label encoder
    label_encoder = dataset_static.get("label_encoder", None)

    # For training, if dynamic_training is True, use UnifiedDataset to inject anomalies on the fly.
    if dynamic_training:
        train_dataset = DynamicSyntheticDataset(
            raw_segments=dataset_static["X_train"],
            raw_labels=dataset_static["y_train"],
            freqs=dataset_static["freqs"],
            engine_config=engine_config,
            mode=mode,
            fault_types_available=fault_types_available,
            peak_segment=peak_segment,
            amplitude_range=amplitude_range,
            normalization_method=normalization_method,
            add_noise=add_noise,
            synthetic_fault_fraction=None  # will default based on mode
        )
    else:
        X_train = torch.tensor(dataset_static["X_train"], dtype=torch.float32).unsqueeze(1)
        y_train = torch.tensor(dataset_static["y_train"], dtype=torch.long)
        train_dataset = TensorDataset(X_train, y_train)

    # Create static validation and test datasets.
    X_val = torch.tensor(dataset_static["X_val"], dtype=torch.float32).unsqueeze(1)
    y_val = torch.tensor(dataset_static["y_val"], dtype=torch.long)
    val_dataset = TensorDataset(X_val, y_val)

    X_test = torch.tensor(dataset_static["X_test"], dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor(dataset_static["y_test"], dtype=torch.long)
    test_dataset = TensorDataset(X_test, y_test)

     # In multiclass mode, you might also want access to the label encoder for inference
    datasets = {"train": train_dataset, "val": val_dataset, "test": test_dataset}
    if mode == "multiclass":
        datasets["label_encoder"] = label_encoder

    return datasets

In [ ]:
from torch.utils.data import DataLoader

# Create unified datasets (with dynamic training)
datasets = create_unified_datasets(
    file_label_map=file_label_map,
    engine_config=engine_config,
    mode="binary",  # or "multiclass"
    dynamic_training=True,
    f_sampling=f_sampling,
    cutoff_freq=cutoff_freq,
    segment_length=segment_length,
    peak_segment=peak_segment,
    step=shift,
    apply_window=False,
    db=True,
    fault_types_available=fault_types_available,
    amplitude_range=amplitude_range,
    normalization_method=normalization_method,
    add_noise=True,
    test_size=0.3,
    val_size=0.2,
    seed=SEED
)

In [ ]:
train_loader_binary = DataLoader(datasets["train"], batch_size=batch_size, shuffle=True)
val_loader_binary   = DataLoader(datasets["val"], batch_size=batch_size, shuffle=False)
test_loader_binary  = DataLoader(datasets["test"], batch_size=batch_size, shuffle=False)

In [ ]:
# Initialize ResNet
model_bin = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for binary classification...")
model_bin = train_resnet_model(model_bin, train_loader_binary, val_loader=val_loader_binary, device=device, num_epochs=num_epochs_binary)

In [ ]:
true_labels_bin, predictions_bin = inference_resnet_model(model_bin, test_loader_binary, device=device)

In [ ]:
# Calculate metrics 
cm_bin, acc_bin, prec_bin, rec_bin, f1_bin = calculate_metrics(true_labels_bin, predictions_bin)

In [ ]:
binary_report = classification_report(true_labels_bin, predictions_bin, target_names=['normal', 'anomalous'], output_dict=True)
binary_report_prefixed = {f"binary_{key}": value for key, value in binary_report.items()}
experiment.log_metrics(binary_report_prefixed)

In [ ]:
fig_bin, ax_bin = plt.subplots(figsize=(5,4))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['normal','anomalous'], yticklabels=['normal','anomalous'], ax=ax_bin)
ax_bin.set_xlabel('Predicted Label')
ax_bin.set_ylabel('True Label')
ax_bin.set_title('Binary Classification - Confusion Matrix')
experiment.log_figure(figure=fig_bin, figure_name="confusion_matrix_binary.png")
plt.close(fig_bin)

## 3. Train the ResNet Multiclass Classifier

In [ ]:
dataset_multi = create_unified_datasets(
    file_label_map=file_label_map,
    engine_config=engine_config,
    mode="multiclass",
    dynamic_training=True,
    f_sampling=f_sampling,
    cutoff_freq=cutoff_freq,
    segment_length=segment_length,
    peak_segment=peak_segment,
    step=shift,
    apply_window=False,
    db=True,
    fault_types_available=fault_types_available,
    amplitude_range=amplitude_range,
    normalization_method=normalization_method,
    add_noise=True,
    test_size=0.3,
    val_size=0.2,
    seed=SEED
)

train_loader_multi = DataLoader(dataset_multi["train"], batch_size=batch_size, shuffle=True)
val_loader_multi = DataLoader(dataset_multi["val"], batch_size=batch_size, shuffle=False)
test_loader_multi = DataLoader(dataset_multi["test"], batch_size=batch_size, shuffle=False)

In [ ]:
# Train ResNet for Multiclass Classification
num_classes_multi = len(dataset_multi['label_encoder'].classes_)
model_multi = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=num_classes_multi, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for multiclass classification...")
model_multi = train_resnet_model(model_multi, train_loader_multi, val_loader=val_loader_multi, device=device, num_epochs=num_epochs_multi)

In [ ]:
true_labels_multi, predictions_multi = inference_resnet_model(model_multi, test_loader_multi, device=device)

In [ ]:
# Calculate metrics 
cm_multi, acc_multi, prec_multi, rec_multi, f1_multi = calculate_metrics(true_labels_multi, predictions_multi)

In [ ]:
multiclass_report = classification_report(true_labels_multi, predictions_multi, target_names=dataset_multi['label_encoder'].classes_, output_dict=True)
multiclass_report_prefixed = {f"multiclass_{key}": value for key, value in multiclass_report.items()}
experiment.log_metrics(multiclass_report_prefixed)

In [ ]:
fig_multi, ax_multi = plt.subplots(figsize=(5,4))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=dataset_multi['label_encoder'].classes_,
            yticklabels=dataset_multi['label_encoder'].classes_, ax=ax_multi)
ax_multi.set_xlabel('Predicted Label')
ax_multi.set_ylabel('True Label')
ax_multi.set_title('Multiclass Classification - Confusion Matrix')
experiment.log_figure(figure=fig_multi, figure_name="confusion_matrix_multiclass.png")
plt.close(fig_multi)

In [ ]:
experiment.end()